In [0]:
for tabela in [
    "tickets",
    "users",
    "tickets_users",
    "ticketsatisfactions",
]:
    spark.table(f"glpi.silver.{tabela}") \
         .createOrReplaceTempView(tabela)

    print(f"Extraindo tabela {tabela}")

In [0]:
d_users = spark.sql("""
SELECT DISTINCT

    user_id,
    login,
    nome_completo

FROM users

WHERE user_id IS NOT NULL
""")

d_status = spark.sql("""
SELECT DISTINCT

    status_id,
    status_descricao

FROM tickets
""")

p_tickets_users = spark.sql("""
SELECT
    ticket_id,
    user_id      AS usuario_id,
    CASE type
        WHEN 1 THEN 'Solicitante'
        WHEN 2 THEN 'Técnico'
        WHEN 3 THEN 'Grupo responsável'
        ELSE 'Outro'
END AS tipo_relacionamento

FROM tickets_users
WHERE user_id IS NOT NULL
""")

f_tickets = spark.sql("""
SELECT
    t.ticket_id,
    t.itilcategories_id,
    t.entities_id,
    t.status_id,
    t.priority,
    t.urgency,
    t.impact,
    t.data_criacao,
    t.data_abertura,
    t.data_modificacao,
    t.data_solucao,
    t.data_fechamento,
    t.horas_resolucao,
    t.titulo,
    t.content
FROM tickets t
""")

f_tickets.createOrReplaceTempView("f_tickets")

f_ticketsatisfactions = spark.sql("""
SELECT

    id                      AS satisfacao_id,
    tickets_id              AS ticket_id,

    satisfaction,

    comment,

    date_begin,
    date_answered

FROM ticketsatisfactions s
                           
INNER JOIN f_tickets t
    ON s.tickets_id = t.ticket_id
""")

tabelas = {
    "f_tickets": f_tickets,
    "f_ticketsatisfactions": f_ticketsatisfactions,
    "d_users": d_users,
    "d_status": d_status,
    "p_tickets_users": p_tickets_users,
    
}

for tabela, df in tabelas.items():
    df.write.mode("overwrite").saveAsTable(f"glpi.gold_chamados.{tabela}")
    
    print(f"{tabela} criada.")
    print(f"{df.count():,} registros\n")

In [0]:
%sql
-- View: Total de Chamados
CREATE OR REPLACE VIEW glpi.gold_chamados.v_total_chamados AS
SELECT COUNT(*) AS total
FROM glpi.gold_chamados.f_tickets;

-- View: Chamados Abertos
CREATE OR REPLACE VIEW glpi.gold_chamados.v_chamados_abertos AS
SELECT COUNT(ticket_id) AS abertos
FROM glpi.gold_chamados.f_tickets
WHERE status_id IN (4, 2, 3);

-- View: Chamados Fechados
CREATE OR REPLACE VIEW glpi.gold_chamados.v_chamados_fechados AS
SELECT COUNT(f.ticket_id) AS fechados
FROM glpi.gold_chamados.f_tickets f
JOIN glpi.gold_chamados.d_status d ON f.status_id = d.status_id
WHERE d.status_descricao = 'Chamado Fechado';

-- View: Tempo Médio de Resolução
CREATE OR REPLACE VIEW glpi.gold_chamados.v_tempo_medio_resolucao AS
SELECT ROUND(AVG(horas_resolucao), 2) AS horas_medias
FROM glpi.gold_chamados.f_tickets
WHERE horas_resolucao IS NOT NULL;

-- View: Tickets Criados por Dia
CREATE OR REPLACE VIEW glpi.gold_chamados.v_tickets_por_dia AS
SELECT
    DATE(data_criacao) AS data,
    COUNT(*) AS quantidade
FROM glpi.gold_chamados.f_tickets
GROUP BY DATE(data_criacao)
ORDER BY data;

-- View: Tickets por Prioridade
CREATE OR REPLACE VIEW glpi.gold_chamados.v_tickets_por_prioridade AS
SELECT
    CASE priority
        WHEN 1 THEN 'Muito Baixa'
        WHEN 2 THEN 'Baixa'
        WHEN 3 THEN 'Média'
        WHEN 4 THEN 'Alta'
        WHEN 5 THEN 'Muito Alta'
        ELSE 'Não definida'
    END AS prioridade,
    COUNT(*) AS quantidade
FROM glpi.gold_chamados.f_tickets
GROUP BY priority
ORDER BY priority;

-- View: Tickets por Status
CREATE OR REPLACE VIEW glpi.gold_chamados.v_tickets_por_status AS
SELECT
    s.status_descricao,
    COUNT(*) AS quantidade
FROM glpi.gold_chamados.f_tickets t
INNER JOIN glpi.gold_chamados.d_status s ON t.status_id = s.status_id
GROUP BY s.status_descricao
ORDER BY quantidade DESC;

-- View: Chamados por Atendente
CREATE OR REPLACE VIEW glpi.gold_chamados.v_chamados_por_atendente AS
SELECT
    u.nome_completo,
    COUNT(*) AS chamados
FROM glpi.gold_chamados.p_tickets_users tu
JOIN glpi.gold_chamados.d_users u ON tu.usuario_id = u.user_id
WHERE tu.tipo_relacionamento = 'Técnico'
GROUP BY u.nome_completo
ORDER BY chamados DESC;

-- View: Satisfação - Positivos e Negativos
CREATE OR REPLACE VIEW glpi.gold_chamados.v_satisfacao AS
SELECT
    ROUND(
        SUM(CASE WHEN satisfaction >= 4 THEN 1 ELSE 0 END) / COUNT(*), 2
    ) AS perc_positivos,
    SUM(CASE WHEN satisfaction >= 4 THEN 1 ELSE 0 END) AS qtd_positivos,
    ROUND(
        SUM(CASE WHEN satisfaction <= 3 THEN 1 ELSE 0 END) / COUNT(*), 2
    ) AS perc_negativos,
    SUM(CASE WHEN satisfaction <= 3 THEN 1 ELSE 0 END) AS qtd_negativos
FROM glpi.gold_chamados.f_ticketsatisfactions
WHERE satisfaction IS NOT NULL;

In [0]:
%sql
-- Total de Chamados
-- SELECT * FROM glpi.gold_chamados.v_total_chamados;

-- Chamados Abertos e Fechados
-- SELECT 
--     (SELECT abertos FROM glpi.gold_chamados.v_chamados_abertos) AS abertos,
--     (SELECT fechados FROM glpi.gold_chamados.v_chamados_fechados) AS fechados;

-- -- Tempo Médio de Resolução
-- SELECT * FROM glpi.gold_chamados.v_tempo_medio_resolucao;

-- -- Tickets Criados por Dia
-- SELECT * FROM glpi.gold_chamados.v_tickets_por_dia;

-- -- Tickets por Prioridade
-- SELECT * FROM glpi.gold_chamados.v_tickets_por_prioridade;

-- -- Tickets por Status
SELECT * FROM glpi.gold_chamados.v_tickets_por_status;

-- -- Chamados por Atendente (Top 10)
-- SELECT * FROM glpi.gold_chamados.v_chamados_por_atendente LIMIT 10;

-- -- Satisfação do Cliente
-- SELECT * FROM glpi.gold_chamados.v_satisfacao;